### In this step, we connect to Google BigQuery and execute an SQL query to extract the dataset for our A/B testing project. The resulting dataset contains all the necessary information about sessions, events, test groups, and dimensions required for further statistical analysis and visualization.

In [ ]:
!pip install --upgrade google-cloud-bigquery

from google.colab import auth
from google.cloud import bigquery
import pandas as pd

# Аутентифікація
auth.authenticate_user()

# Створення клієнта для BigQuery
client = bigquery.Client(project="data-analytics-mate")

# SQL-запит
query = """
WITH
 session_info AS (
   SELECT
     s.date,
     s.ga_session_id,
     sp.country,
     sp.device,
     sp.continent,
     sp.channel,
     ab.test AS test_number,
     ab.test_group
   FROM `DA.ab_test` ab
   JOIN `DA.session` s
     ON ab.ga_session_id = s.ga_session_id
   JOIN `DA.session_params` sp
     ON sp.ga_session_id = ab.ga_session_id
 ),
 session_with_orders AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group,
     COUNT(DISTINCT o.ga_session_id) AS session_with_orders
   FROM `DA.order` o
   JOIN session_info
     ON o.ga_session_id = session_info.ga_session_id
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group
 ),
 event AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group,
     sp.event_name,
     COUNT(sp.ga_session_id) AS event_cnt
   FROM `DA.event_params` sp
   JOIN session_info
     ON sp.ga_session_id = session_info.ga_session_id
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group,
     sp.event_name
 ),
 session AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group,
     COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
   FROM session_info
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group
 ),
 account AS (
   SELECT
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group,
     COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
   FROM `DA.account_session` acs
   JOIN session_info
     ON acs.ga_session_id = session_info.ga_session_id
   GROUP BY
     session_info.date,
     session_info.country,
     session_info.device,
     session_info.continent,
     session_info.channel,
     session_info.test_number,
     session_info.test_group
 )
SELECT
 session_with_orders.date,
 session_with_orders.country,
 session_with_orders.device,
 session_with_orders.continent,
 session_with_orders.channel,
 session_with_orders.test_number,
 session_with_orders.test_group,
 'session_with_orders' AS event_name,
 session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
 event.date,
 event.country,
 event.device,
 event.continent,
 event.channel,
 event.test_number,
 event.test_group,
 event.event_name,
 event.event_cnt AS value
FROM event
UNION ALL
SELECT
 session.date,
 session.country,
 session.device,
 session.continent,
 session.channel,
 session.test_number,
 session.test_group,
 'session' AS event_name,
 session_cnt AS value
FROM session
UNION ALL
SELECT
 account.date,
 account.country,
 account.device,
 account.continent,
 account.channel,
 account.test_number,
 account.test_group,
 'new_account' AS event_name,
 new_account_cnt AS value
FROM account;

"""

# Виконання запиту
query_job = client.query(query)  # Виконання SQL-запиту
results = query_job.result()  # Очікування завершення запиту

# Перетворення результатів на DataFrame
df = results.to_dataframe(create_bqstorage_client=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Шлях до папки на Google Drive
output_path = '/content/drive/MyDrive/Portfolio_2_A_B_Testing/portfolio2_main.csv'

# Зберегти DataFrame
df.to_csv(output_path, index=False)

print(f'Файл успішно збережено:\n{output_path}')

Файл успішно збережено:
/content/drive/MyDrive/Portfolio_2_A_B_Testing/portfolio2_main.csv


### To ensure the dataset is suitable for further analysis, we perform a basic data quality assessment. We examine the dataset structure, data types, descriptive statistics, and check for missing values.

In [ ]:
df.head()

,date,country,device,continent,channel,test_number,test_group,event_name,value
0,2020-11-01,Lithuania,mobile,Europe,Organic Search,2,2,new_account,1
1,2020-11-01,El Salvador,desktop,Americas,Social Search,2,1,new_account,1
2,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new_account,1
3,2020-11-01,Lithuania,desktop,Europe,Paid Search,2,2,new_account,1
4,2020-11-02,North Macedonia,desktop,Europe,Direct,2,1,new_account,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   date         800996 non-null  dbdate
 1   country      800996 non-null  object
 2   device       800996 non-null  object
 3   continent    800996 non-null  object
 4   channel      800996 non-null  object
 5   test_number  800996 non-null  Int64 
 6   test_group   800996 non-null  Int64 
 7   event_name   800996 non-null  object
 8   value        800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


In [ ]:
df.describe()

,test_number,test_group,value
count,800996.0,800996.0,800996.0
mean,2.831191,1.499608,9.475863
std,1.11694,0.5,37.493267
min,1.0,1.0,1.0
25%,2.0,1.0,1.0
50%,3.0,1.0,2.0
75%,4.0,2.0,5.0
max,4.0,2.0,1575.0


## **Conclusion**

### The dataset was successfully retrieved from BigQuery and loaded into a Pandas DataFrame. Initial data validation confirmed that the dataset contains all required columns, has no missing values, and is ready for further conversion rate calculations and statistical significance analysis.

## **Conversion Rate Calculation and Statistical Analysis**

### In this step, we define the conversion metrics used to evaluate the performance of different A/B test variations. Each metric represents a specific stage of the e-commerce conversion funnel and measures the percentage of users who successfully completed a particular action. To make the analysis scalable, all metrics will be calculated using reusable functions and iteration, allowing new metrics to be added with minimal code changes.

## Conversion Metrics

### The following six conversion metrics will be calculated:

1. **Begin Checkout Conversion Rate** (`begin_checkout / session`)
   - Measures the percentage of sessions that reached the checkout process.
   - Indicates users' intention to complete a purchase.

2. **Shipping Information Conversion Rate** (`add_shipping_info / session`)
   - Measures the percentage of sessions where users entered their shipping information.
   - Evaluates progress through the checkout funnel.

3. **Payment Information Conversion Rate** (`add_payment_info / session`)
   - Measures the percentage of sessions where users entered payment details.
   - Indicates how many users reached the payment step.

4. **New Account Conversion Rate** (`new_account / session`)
   - Measures the percentage of sessions that resulted in a new account registration.
   - Reflects user engagement and long-term customer acquisition.

5. **Purchase Conversion Rate** (`session_with_orders / session`)
   - Measures the percentage of sessions that resulted in a completed order.
   - Represents the primary business KPI for the e-commerce website.

   ### Approach

### To improve code readability and maintainability, the conversion metrics will be calculated using reusable functions and simple iteration. This approach minimizes code duplication while keeping the analysis easy to understand and extend with additional metrics if needed.

In [ ]:
sorted(df['event_name'].unique())

['add_payment_info',
 'add_shipping_info',
 'add_to_cart',
 'begin_checkout',
 'click',
 'first_visit',
 'new_account',
 'page_view',
 'scroll',
 'select_item',
 'select_promotion',
 'session',
 'session_start',
 'session_with_orders',
 'user_engagement',
 'view_item',
 'view_item_list',
 'view_promotion',
 'view_search_results']

In [ ]:
df['event_name'].value_counts()

,count
event_name,
session,107210
session_start,106242
page_view,101907
user_engagement,94520
first_visit,81621
scroll,73643
view_promotion,61695
view_item,44869
session_with_orders,25892


### Metrics Configuration

To make the analysis flexible and avoid repetitive code, all conversion metrics and analysis dimensions are stored in configuration objects. Instead of writing separate calculations for each metric and dimension, the program will iterate through these configurations and perform the same calculations automatically.

This approach reduces code duplication, improves readability, and makes it easy to extend the analysis by simply adding new metrics or dimensions.

In [ ]:
metrics = [
    {
        'metric': 'begin_checkout/session',
        'numerator': 'begin_checkout',
        'denominator': 'session'
    },
    {
        'metric': 'add_shipping_info/session',
        'numerator': 'add_shipping_info',
        'denominator': 'session'
    },
    {
        'metric': 'add_payment_info/session',
        'numerator': 'add_payment_info',
        'denominator': 'session'
    },
    {
        'metric': 'new_account/session',
        'numerator': 'new_account',
        'denominator': 'session'
    },
    {
        'metric': 'session_with_orders/session',
        'numerator': 'session_with_orders',
        'denominator': 'session'
    }
]

### Z-test Function

This function performs a two-proportion Z-test to compare the conversion rates between the Control Group and the Test Group. It returns the Z-statistic, p-value, and a Boolean flag indicating whether the difference is statistically significant at α = 0.05.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
def run_z_test(
    control_conversions,
    control_sessions,
    test_conversions,
    test_sessions
):

    counts = [
        test_conversions,
        control_conversions
    ]

    observations = [
        test_sessions,
        control_sessions
    ]

    z_stat, p_value = proportions_ztest(
        count=counts,
        nobs=observations
    )

    significant = p_value < 0.05

    return z_stat, p_value, significant

### Conversion Calculation Function

This function calculates the conversion rate for a selected metric within a specific A/B test. It compares the Control and Test groups, calculates the relative metric change, performs the statistical significance test, and returns all calculated results as a dictionary.

In [ ]:
def calculate_conversion(df, test_number, metric):

    # Дані одного тесту
    test_df = df[df['test_number'] == test_number]

    # Group 1 = Control
    control_df = test_df[test_df['test_group'] == 1]

    # Group 2 = Test
    test_df_group = test_df[test_df['test_group'] == 2]

    # ---------- Control ----------
    control_conversions = control_df.loc[
        control_df['event_name'] == metric['numerator'],
        'value'
    ].sum()

    control_sessions = control_df.loc[
        control_df['event_name'] == metric['denominator'],
        'value'
    ].sum()

    control_cr = control_conversions / control_sessions

    # ---------- Test ----------
    test_conversions = test_df_group.loc[
        test_df_group['event_name'] == metric['numerator'],
        'value'
    ].sum()

    test_sessions = test_df_group.loc[
        test_df_group['event_name'] == metric['denominator'],
        'value'
    ].sum()

    test_cr = test_conversions / test_sessions

    # ---------- Metric Change ----------
    metric_change = ((test_cr - control_cr) / control_cr) * 100

    # Statistical significance
    z_stat, p_value, significant = run_z_test(
    control_conversions,
    control_sessions,
    test_conversions,
    test_sessions
    )

    return {
    'test_number': test_number,
    'metric': metric['metric'],
    'numerator_event': metric['numerator'],
    'denominator_event': metric['denominator'],

    'numerator_count_test': test_conversions,
    'denominator_count_test': test_sessions,
    'conversion_rate_test': test_cr,

    'numerator_count_control': control_conversions,
    'denominator_count_control': control_sessions,
    'conversion_rate_control': control_cr,

    'metric_change': metric_change,

    'z_stat': z_stat,
    'p_value': p_value,
    'significant': significant
    }

### Function Validation

Before processing all tests and metrics, the function is executed for a single metric from **Test 1**. This validation step helps verify that the calculations and returned values are correct before running the full analysis.

In [ ]:
results = []

In [ ]:
result = calculate_conversion(df, 1, metrics[0])

result

{'test_number': 1,
 'metric': 'begin_checkout/session',
 'numerator_event': 'begin_checkout',
 'denominator_event': 'session',
 'numerator_count_test': np.int64(4021),
 'denominator_count_test': np.int64(45193),
 'conversion_rate_test': np.float64(0.08897395614365057),
 'numerator_count_control': np.int64(3784),
 'denominator_count_control': np.int64(45362),
 'conversion_rate_control': np.float64(0.08341783871963317),
 'metric_change': np.float64(6.660586643453422),
 'z_stat': np.float64(2.978782961833959),
 'p_value': np.float64(0.0028939568500496436),
 'significant': np.True_}

In [ ]:
results = []

for test_number in [1, 2, 3, 4]:
    for metric in metrics:

        result = calculate_conversion(
            df,
            test_number,
            metric
        )

        results.append(result)

In [ ]:
statistics_df

,test_number,metric,numerator_event,denominator_event,numerator_count_test,denominator_count_test,conversion_rate_test,numerator_count_control,denominator_count_control,conversion_rate_control,metric_change,z_stat,p_value,significant
0,1,begin_checkout/session,begin_checkout,session,4021,45193,0.088974,3784,45362,0.083418,6.660587,2.978783,0.002894,True
1,1,add_shipping_info/session,add_shipping_info,session,3221,45193,0.071272,3034,45362,0.066884,6.560481,2.603571,0.009226,True
2,1,add_payment_info/session,add_payment_info,session,2229,45193,0.049322,1988,45362,0.043825,12.542021,3.924884,0.000087,True
3,1,new_account/session,new_account,session,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,False
4,1,session_with_orders/session,session_with_orders,session,4526,45193,0.100148,4514,45362,0.099511,0.640785,0.320049,0.748931,False
5,2,begin_checkout/session,begin_checkout,session,4313,50244,0.085841,4262,50637,0.084168,1.988164,0.952898,0.340642,False
6,2,add_shipping_info/session,add_shipping_info,session,3510,50244,0.069859,3480,50637,0.068724,1.650995,0.709557,0.477979,False
7,2,add_payment_info/session,add_payment_info,session,2409,50244,0.047946,2344,50637,0.046290,3.576911,1.240994,0.214608,False
8,2,new_account/session,new_account,session,4184,50244,0.083274,4165,50637,0.082252,1.241934,0.588793,0.556000,False
9,2,session_with_orders/session,session_with_orders,session,5003,50244,0.099574,5102,50637,0.100756,-1.173410,-0.625388,0.531717,False


### Export Results

The final statistical results are exported to a CSV file. This dataset will be used later in Tableau to build the dashboard and visualize the outcomes of the A/B tests.

In [ ]:
output_path = '/content/drive/MyDrive/Portfolio_2_A_B_Testing/portfolio2_statistics.csv'

statistics_df.to_csv(output_path, index=False)

print(f'Файл успішно збережено:\n{output_path}')

Файл успішно збережено:
/content/drive/MyDrive/Portfolio_2_A_B_Testing/portfolio2_statistics.csv


## **Conclusion**

The statistical analysis was conducted to evaluate conversion rates, relative metric changes, and statistical significance for five conversion metrics across four independent A/B tests.

For each A/B test, the **Control Group** was compared with the **Test Group** using a two-proportion Z-test with a significance level of **α = 0.05**. Each conversion metric was evaluated independently to determine whether the observed differences were statistically significant.

### Key Findings

- **Test 1**
  - Statistically significant improvements were observed for:
    - **begin_checkout/session**
    - **add_shipping_info/session**
    - **add_payment_info/session**
  - No statistically significant differences were found for:
    - **new_account/session**
    - **session_with_orders/session**

- **Test 2**
  - No statistically significant differences were observed across any of the analyzed conversion metrics.

- **Test 3**
  - A statistically significant improvement was found only for:
    - **begin_checkout/session**
  - All remaining conversion metrics were not statistically significant.

- **Test 4**
  - Statistically significant improvements were observed for:
    - **begin_checkout/session**
    - **new_account/session**
  - The remaining conversion metrics were not statistically significant.

### Overall Conclusion

The results show that the effectiveness of an A/B test should be evaluated for each conversion metric separately rather than for the experiment as a whole. Within the same A/B test, some metrics demonstrated statistically significant improvements, while others showed no significant differences between the Control and Test groups.

These findings suggest that experimental changes may affect different stages of the conversion funnel differently. Therefore, business decisions should be based on the statistical significance of individual conversion metrics rather than on the overall outcome of the experiment.

## Interactive Dashboard (Tableau Public):

### https://public.tableau.com/views/Portfolio2_17852343207190/ABTestAnalysisDistributionStatisticalSignificance?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link


## CSV File with Statistical Results

### https://drive.google.com/file/d/1-3pJG8kdTXSXqGpMO8ws4T-yeSYWD1np/view?usp=drive_link